In [0]:
%sql
CREATE CATALOG IF NOT EXISTS opensky
MANAGED LOCATION 
'abfss://batch@uralibootcamp.dfs.core.windows.net/managed/opensky';

CREATE SCHEMA IF NOT EXISTS opensky.bronze;

CREATE SCHEMA IF NOT EXISTS  opensky.silver;

CREATE SCHEMA IF NOT EXISTS  opensky.gold;

In [0]:
raw_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/raw/Flights"


schema_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/schema/bronze"


checkpoint_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/checkpoint/bronze"

Purpose:

Incrementally load new files using Auto Loader.

In [0]:
from pyspark.sql.functions import *


bronze_df = (

spark.readStream
.format("cloudFiles")
.option(
"cloudFiles.format",
"csv"
)
.option(
"cloudFiles.inferColumnTypes",
"true"
)
.option(
"cloudFiles.schemaEvolutionMode",
"addNewColumns"
)
.option(
"cloudFiles.schemaLocation",
schema_path
)
.load(raw_path)

)

Add Metadata Columns
Very important for history tracking.

In [0]:
bronze_df = (

bronze_df

.withColumn(
"source_file",
input_file_name()
)

.withColumn(
"bronze_timestamp",
current_timestamp()
)

)

Write Bronze Delta Table

In [0]:
bronze_df.writeStream \
.format("delta") \
.option(
    "checkpointLocation",
    checkpoint_path
) \
.outputMode("append") \
.trigger(
    availableNow=True
) \
.table(
    "opensky.bronze.bronze_flights"
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://batch@uralibootcamp.dfs.core.windows.net/raw/Flights"
    )
)